# Import Modules

In [113]:
import random
import requests
import numpy as np
import folium
import geopandas as gpd
import pandas as pd
import branca.colormap as cm

# Get schools via Overpass API

In [45]:
def get_bradford_secondary_schools():
    url = "https://overpass-api.de/api/interpreter"
    headers = {
        "User-Agent": "BradfordSchoolMapper/1.0 (hal@kolb.com)"
    }

    # Bounding box for Bradford
    bbox = "53.71,-2.05,53.96,-1.65"
    bbox = "53.61,-2.15,54.06,-1.55"
    bbox = "53.51,-2.25,54.16,-1.45"
    
    query = f"""
    [out:json][timeout:60];
    (
      nwr["amenity"="school"]["isced:level"~"2|3"]({bbox});
      nwr["amenity"="school"]["school:type"="secondary"]({bbox});
    );
    out center;
    """
    
    try:
        response = requests.get(url, params={'data': query}, headers=headers, timeout=60)
        response.raise_for_status()
        data = response.json()
        elements = data.get('elements', [])

        # Process into a list of dictionaries for Pandas
        school_list = []
        for e in elements:
            tags = e.get('tags', {})
            school_list.append({
                "Name": tags.get('name', 'Unnamed School'),
                "URN": tags.get('ref:GB:uass') or tags.get('ref:edubase'),
                "Postcode": tags.get('addr:postcode'),
                "Lat": e.get('lat') or e.get('center', {}).get('lat'),
                "Lon": e.get('lon') or e.get('center', {}).get('lon'),
                "Website": tags.get('website')
            })

        # Create DataFrame
        df = pd.DataFrame(school_list)
        return df

    except Exception as e:
        print(f"Error fetching data: {e}")
        return pd.DataFrame()

# 1. Get the OSM data
df_schools = get_bradford_secondary_schools()

# 2. Example: How to merge Attainment 8 data
# Note: You would download '2023-2024_england_ks4_final.csv' from the Gov.uk site
# attainment_df = pd.read_csv('government_performance_data.csv')
# df_final = pd.merge(df_schools, attainment_df[['URN', 'ATT8SCR']], on='URN', how='left')

# 3. Export to CSV
if not df_schools.empty:
    df_schools.to_csv("bradford_schools.csv", index=False)
    print("Success! Data exported to bradford_schools.csv")
    print(df_schools.head())

Success! Data exported to bradford_schools.csv
                          Name     URN  Postcode        Lat       Lon  \
0     Hazelbeck Special School  139977  BD16 1EE  53.841401 -1.827540   
1        Brian Jackson College  132732  WF16 0AD  53.708754 -1.667765   
2      The Springboard Project  145922   OL1 1AN  53.539407 -2.113795   
3      North Ridge High School  132905    M9 0RP  53.537068 -2.220648   
4  Our Lady’s R.C. High School  105576    M9 0RP  53.537556 -2.220533   

                                     Website  
0                 https://www.hazelbeck.org/  
1                                       None  
2        https://www.springboardproject.org/  
3  https://www.northridge.manchester.sch.uk/  
4        https://www.olhs-manchester.org.uk/  


# Combine school data 

In [63]:
school_data = pd.read_csv("bradford_schools.csv")
school_perf = pd.read_csv("key-stage-4-performance_2024-25/data/202425_performance_tables_schools_revised.csv")

school_perf.loc[school_perf["breakdown_topic"] == "Total"][["school_urn","attainment8_average", "gcse_91_percent"]]

,school_urn,attainment8_average,gcse_91_percent
0,100544,7.9,66.7
7,100001,49.7,100
14,100003,39.2,98.6
21,137181,45.9,97.8
28,100049,41.5,96.6
...,...,...,...
40250,112461,z,z
40257,134781,c,c
40264,135940,42.4,96.4
40271,141993,c,c


In [64]:
school_data["Attainment 8"] = pd.NA
school_data["Pupil Count"] = pd.NA

for i in range(len(school_data)):
    urn = school_data.loc[i, "URN"]
    att8 = school_perf.loc[(school_perf["school_urn"] == urn) & (school_perf["breakdown_topic"] == "Total")]["attainment8_average"]
    pupil_count = school_perf.loc[(school_perf["school_urn"] == urn) & (school_perf["breakdown_topic"] == "Total")]["pupil_count"]
    try:
        att8 = att8.item()
    except ValueError:
        att8 = pd.NA
    try:
        pupil_count = pupil_count.item()
    except ValueError:
        pupil_count = pd.NA

    try:
        att8 = float(att8)
    except:
        att8 = pd.NA
    try:
        pupil_count = int(pupil_count)
    except:
        pupil_count = pd.NA
    school_data.loc[i, "Attainment 8"] = att8
    school_data.loc[i, "Pupil Count"] = pupil_count

school_data

,Name,URN,Postcode,Lat,Lon,Website,Attainment 8,Pupil Count
0,Hazelbeck Special School,139977.0,BD16 1EE,53.841401,-1.827540,https://www.hazelbeck.org/,<NA>,23
1,Brian Jackson College,132732.0,WF16 0AD,53.708754,-1.667765,NaN,0.6,34
2,The Springboard Project,145922.0,OL1 1AN,53.539407,-2.113795,https://www.springboardproject.org/,0.1,38
3,North Ridge High School,132905.0,M9 0RP,53.537068,-2.220648,https://www.northridge.manchester.sch.uk/,<NA>,30
4,Our Lady’s R.C. High School,105576.0,M9 0RP,53.537556,-2.220533,https://www.olhs-manchester.org.uk/,49.1,207
...,...,...,...,...,...,...,...,...
263,Outwood Grange Academy,135961.0,WF1 2PF,53.706870,-1.512497,https://www.grange.outwood.com/,47.2,342
264,Wakefield Girls' High School,108305.0,WF1 2QS,53.690057,-1.503117,https://wgsf.org.uk/wghs/,61.7,108
265,Queen Elizabeth Grammar School,108306.0,NaN,53.691876,-1.498084,https://wgsf.org.uk/qegs/,43.8,96
266,Hulme Grammar School,105745.0,OL8 4BX,53.529706,-2.122338,https://hulmegrammar.org/,59.3,101


# LSOA house price data

In [48]:
lsoa_house_price = pd.read_csv("C:\\Users\\halko\\Downloads\\hpssadataset46medianpricepaidforresidentialpropertiesbylsoa\\lsoa_house_prices.csv")
lsoa_house_price['Price Mar 2023'] = pd.to_numeric(
    lsoa_house_price['Price Mar 2023'].str.replace(',', ''), 
    errors='coerce'
)
lsoa_house_price

,LSOA code,Price Mar 2023
0,E01011949,106500.0
1,E01011950,43500.0
2,E01011951,66000.0
3,E01011952,60000.0
4,E01011953,92500.0
...,...,...
34748,W01001320,149500.0
34749,W01001321,120000.0
34750,W01001322,136000.0
34751,W01001324,145000.0


In [ ]:
# 1. Load and Clean House Price Data
lsoa_house_price = pd.read_csv("lsoa_house_prices.csv")

# Remove commas and convert to numeric; errors='coerce' handles missing values
lsoa_house_price['Price Mar 2023'] = pd.to_numeric(
    lsoa_house_price['Price Mar 2023'].astype(str).str.replace(',', ''), 
    errors='coerce'
)

# 2. Load and Filter Geometries
file_path = "Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BSC_V4_6894679968818356315.geojson"
gdf = gpd.read_file(file_path)
gdf = gdf[(gdf['LSOA21NM'].str.contains('Bradford')) | (gdf['LSOA21NM'].str.contains('Leeds')) | (gdf['LSOA21NM'].str.contains('Calderdale')) | (gdf['LSOA21NM'].str.contains('Kirklees'))].copy() # Calderdale

# 3. Merge and Coordinate Management
# Merge price data onto the GeoDataFrame
merged_gdf = gdf.merge(
    lsoa_house_price, 
    left_on='LSOA21CD', 
    right_on='LSOA code', 
    how='left'
)

# Convert to WGS84 and simplify the merged object
if merged_gdf.crs != 'EPSG:4326':
    merged_gdf = merged_gdf.to_crs(epsg=4326)

merged_gdf['geometry'] = merged_gdf['geometry'].simplify(tolerance=0.0001, preserve_topology=True)

In [50]:
# 4. Create Color Scale
# Filter out NaNs for the colormap calculation
valid_prices = merged_gdf['Price Mar 2023'].dropna()
colormap = cm.LinearColormap(
    colors=['yellow', 'orange', 'red'],
    vmin=valid_prices.min(),
    vmax=valid_prices.quantile(0.95) # Cap at 95th percentile to maintain color contrast
)

# 5. Initialize the Map
m = folium.Map(location=[53.79, -1.75], zoom_start=11, tiles='CartoDB positron')

# 6. Add LSOA Polygons with Dynamic Styling
folium.GeoJson(
    merged_gdf,
    name='Bradford House Prices',
    style_function=lambda feature: {
        'fillColor': colormap(feature['properties']['Price Mar 2023']) 
                     if feature['properties']['Price Mar 2023'] is not None else '#808080',
        'color': 'gray',
        'weight': 0.5,
        'fillOpacity': 0.4,
    },
    # highlight_function=lambda feature: {
    #     'weight': 3,
    #     'fillOpacity': 0.6,
    # },
    # tooltip=folium.GeoJsonTooltip(
    #     fields=['LSOA21NM', 'LSOA21CD', 'Price Mar 2023'],
    #     aliases=['LSOA Name:', 'Code:', 'Median Price (£):'],
    #     localize=True
    # )
).add_to(m)

# 7. Add the Colormap Legend
colormap.caption = 'Median House Price (March 2023)'
colormap.add_to(m)

# # 8. Add School Markers (Existing logic)
# for i in range(len(school_data)):
#     school = school_data.loc[i]
#     name = school["Name"]
#     lat = school["Lat"]
#     lon = school["Lon"]
#     att8 = school["Attainment 8"]
#     pup_count = school["Pupil Count"]
    
#     folium.Marker(
#         location=[lat, lon],
#         popup=folium.Popup(name, max_width=300),
#         tooltip=f"{name}, Att8: {att8}, pup_count: {pup_count}",
#         icon=folium.Icon(color='blue', icon='book', prefix='fa')
#     ).add_to(m)

In [51]:
import matplotlib.cm as mplcm
import matplotlib.colors as colors
import pandas as pd

# 1. Define the range for your color scale
# We use the actual min/max from your data to ensure the full spectrum is used

vmin = school_data["Attainment 8"].min()
vmax = school_data["Attainment 8"].max()

# 2. Create a normalization object and the colormap
norm = colors.Normalize(vmin=vmin, vmax=vmax)
colormap_schools = mplcm.viridis  # You can also use 'plasma', 'magma', or 'inferno'

def get_att8_color(score):
    if pd.isna(score):
        return 'gray'
    
    # Map the score to a 0.0 - 1.0 range
    normalized_score = norm(score)
    
    # Get the RGBA color from the colormap
    rgba = colormap_schools(normalized_score)
    
    # Convert RGBA to Hex format (which Folium prefers)
    return colors.rgb2hex(rgba)

In [52]:
for i in range(len(school_data)):
    school = school_data.loc[i]
    name = school["Name"]
    lat = school["Lat"]
    lon = school["Lon"]
    att8 = school["Attainment 8"]
    pup_count = school["Pupil Count"]
    
    # 3. Scale the pupil count for the radius 
    # (e.g., 1000 pupils / 100 = 10 pixel radius)
    if pd.isna(pup_count) or pup_count == 0:
        marker_radius = 5 # Default fallback size
    else:
        marker_radius = int(pup_count) / 20
        
    try:
        att8 = float(att8)
    except:
        att8 = pd.NA

    # 4. Get the color for this specific school
    marker_color = get_att8_color(att8)

    # 5. Add the CircleMarker to the map
    folium.CircleMarker(
        location=[lat, lon],
        radius=marker_radius,
        popup=folium.Popup(name, max_width=300),
        tooltip=f"{name} | Att8: {att8} | Pupils: {pup_count}",
        color=marker_color,         # Outline color
        weight=1,                   # Outline thickness
        fill=True,
        fill_color=marker_color,    # Inside color
        fill_opacity=0.9            # Slight transparency to see overlapping schools
    ).add_to(m)

In [53]:
# 9. Save and View
m.save('bradford_house_prices_schools.html')

# Navigation via OSRM with local host using docker image

https://github.com/project-osrm/osrm-backend/pkgs/container/osrm-backend

```
# Extract the graph (using the car profile as an example)
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-extract -p /opt/car.lua /data/west-yorkshire-260414.osm.pbf

# Partition and customize the graph for routing
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-partition /data/west-yorkshire-260414.osrm
docker run -t -v "${PWD}:/data" osrm/osrm-backend osrm-customize /data/west-yorkshire-260414.osrm

# Start local server
docker run -t -i -p 5000:5000 -v "${PWD}:/data" osrm/osrm-backend osrm-routed --algorithm mld /data/west-yorkshire-260414.osrm
```

- Take a given LSOA
- For that LSOA get all OAs within it
- Calculate area (km^2) of each OA
- Get population density of OA from ONS census data
- Calculate population of each OA
- Select a sample OA weighted by population
- Within that OA select a random location contained within the polygon (maybe there's a better way ?)
- For that point use OSRM to calculate the duration of walking, cycling and driving between the point and each school
- Repeat sampling to get a distribution of journey times accross the LSOA

In [129]:
def time_journey(origin, dest, travel_type="driving"):
    # Hit local OSRM server
    url = f"http://127.0.0.1:500{int(travel_type=="foot")}/route/v1/{travel_type}/{origin};{dest}?overview=false"
    response = requests.get(url).json()

    journey_time_seconds = response['routes'][0]['duration']
    return journey_time_seconds

In [29]:
# Load OA population data
oa_population = pd.read_csv("OutputArea_Population_21.csv")
oa_population['Population'] = pd.to_numeric(
    oa_population['Population'].astype(str).str.replace(',', ''), 
    errors='coerce'
)

In [33]:
oa_gdf = gpd.read_file("Output_Areas_2021_EW_BGC_V2.geojson")

In [ ]:
chosen_LSOA = "E01010572"

In [115]:
def sample_start_point(LSOA:str) -> str:
    chosen_oas = oa_population.loc[oa_population["LSOA21CD"] == chosen_LSOA]
    oa_pick = random.choices(list(chosen_oas["OA21CD"]), weights=chosen_oas["Population"], k=1)[0]
    lat = oa_gdf.loc[oa_gdf["OA21CD"] == oa_pick]["LAT"].item()
    long = oa_gdf.loc[oa_gdf["OA21CD"] == oa_pick]["LONG"].item()
    start = f"{long},{lat}"

    return start

In [ ]:
num_samples = 100

for i in range(len(school_data)):
    name = school_data.loc[i, "Name"]
    lat = school_data.loc[i, "Lat"]
    lon = school_data.loc[i, "Lon"]
    dest = f"{lon},{lat}"

    for mode in ["foot", "driving"]:
        times_list = []
        for j in range(num_samples):
            start = sample_start_point(chosen_LSOA)
            times_list.append(time_journey(start, dest, travel_type=mode))
        jour_time = np.array(times_list).mean()

        print(f"Mean {mode} time from {chosen_LSOA} to {name}: {round(jour_time//60):02d}m{round(jour_time%60):02d}s")
    print()

Mean foot time from E01010572 to Hazelbeck Special School: 69m26s
Mean driving time from E01010572 to Hazelbeck Special School: 08m57s
Mean foot time from E01010572 to Brian Jackson College: 230m22s
Mean driving time from E01010572 to Brian Jackson College: 28m23s
Mean foot time from E01010572 to The Springboard Project: 466m36s
Mean driving time from E01010572 to The Springboard Project: 42m38s
Mean foot time from E01010572 to North Ridge High School: 474m57s
Mean driving time from E01010572 to North Ridge High School: 39m01s
Mean foot time from E01010572 to Our Lady’s R.C. High School: 474m57s


In [125]:
import requests

# Example: Origin (Output Area centroid), Destination (School)
# Coordinates must be Longitude, Latitude
origin = "-1.759,53.795" # Bradford approx
dest = "-1.765,53.801"

# Hit your local OSRM server
url = f"http://127.0.0.1:5000/route/v1/driving/{origin};{dest}?overview=false"
response = requests.get(url).json()

journey_time_seconds = response['routes'][0]['duration']
print(f"Journey time: {journey_time_seconds:.2f} seconds")

Journey time: 142.50 seconds
